In [ ]:
import pandas as pd 
import geopandas as gpd
import numpy as np
import requests

from shapely.geometry import LineString, Point
from shapely.affinity import translate

# Make api request

In [ ]:
# Define the base URL and parameters
base_url = "https://maps.udot.utah.gov/central/rest/services/TrafficAndSafety/UDOT_Speed_Limits/MapServer/0/query"
params = {
    "where": "Name='0210'",
    "outFields": "*",
    "f": "json"
}

# Make the GET request
response = requests.get(base_url, params=params)

# Check if the request was successful
if response.status_code == 200:
    data  = response.json()
    features = data['features']
    # Process the data as needed
else:
    print(f"Error: {response.status_code}")

# Convert to a gdf of points

In [ ]:
geoms = []
speed_limits = []
# this loops through the api request and gets the paths and speed limits
for feature in features:
    paths = feature["geometry"]["paths"]
    for path in paths:
        geoms.append(LineString(path))
        speed_limits.append(feature["attributes"]["Speed_Limit"])

# saves the data in a gdf 
gdf = gpd.GeoDataFrame({"speed_limit": speed_limits}, geometry=geoms, crs="EPSG:26912").to_crs('EPSG:4326')

# i drop the snowbird loop
gdf = gdf.drop(1).reset_index(drop=True) 

# reorder the rows so that the points are in order
gdf = gdf.reindex([3, 2, 1, 0]).reset_index(drop=True)

# now explode the coords
exploded = gdf.copy()
exploded["geometry"] = exploded["geometry"].apply(lambda line: list(line.coords))
# Explode into rows (flatten list of coordinates)
exploded = exploded.explode("geometry").reset_index(drop=True)
# Convert each coordinate into a Point geometry
exploded["geometry"] = exploded["geometry"].apply(lambda coord: Point(coord))
# Convert to GeoDataFrame with same CRS and keep speed limits
w_speed_limits = gpd.GeoDataFrame(exploded, geometry="geometry", crs=gdf.crs)

w_speed_limits['linked_coord'] = None
w_speed_limits.head(3)

# Create down lane

In [ ]:
# reverse the gdf 
reverse_w_speed_limits = w_speed_limits[::-1].to_crs(epsg=32612)
# shift the road over
reverse_w_speed_limits["geometry"] = reverse_w_speed_limits["geometry"].apply(lambda geom: translate(geom, xoff=300, yoff=500)).to_crs(epsg=4326)
# concat them together
full_road = pd.concat([w_speed_limits, reverse_w_speed_limits]).reset_index(drop=True)



# adding a col that links the up and down lanes 
full_road['linked_coord'] = full_road.geometry[::-1 ].reset_index(drop=True)



# Outputs

In [ ]:
w_speed_limits.to_parquet("data/roads/hw210_w_speed_limits.parquet", index=False)
w_speed_limits.head()


In [ ]:
full_road.to_parquet("data/roads/hw210_full_road.parquet", index=False)

full_road.head()